### Project - Airline AI Assistant
We'll now bring together what we've learned to make an AI Customer Support 
assistant for an Airline

In [51]:
# Imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [52]:
# Initialization

load_dotenv(override= True)

openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key:
    print(f"OpenAI API Key exists and it begins with {openai_api_key[:7]}....")
else:
    print("OpenAI API Key is not set.")

openai_client = OpenAI()
gpt_model = "gpt-4.1-mini"

# As an alternative, if you'd like to use Ollama instead of OpenAI
# Check that Ollama is running for you locally (see week1/day2 exercise) then uncomment these next 2 lines
# MODEL = "llama3.2"
# openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

OpenAI API Key exists and it begins with sk-proj....


In [53]:
system_message = """
You are an helpful assistant for an airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [54]:
def chat(message, history):

    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]

    response = openai_client.chat.completions.create(
        model= gpt_model,
        messages= messages
    )

    return response.choices[0].message.content

# gr.ChatInterface(fn= chat).launch()

### Tools
Tools are an incredibly powerful feature provided by the frontier LLMs.  

With tools, you can write a function, and have the LLM call that function as 
part of its response.  

Sounds almost spooky... we're giving it the power to run code on our machine?  
Well, kinda.

In [55]:
# Let's start by making a useful function

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    return f"The price of a ticket to {destination_city} is {price}"
    # return json.dumps({"city": city, "price": 799})


In [56]:
get_ticket_price("London")

Tool called for city London


'The price of a ticket to London is $799'

In [57]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [58]:
tools = [{"type": "function", "function": price_function}]

In [59]:
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

### Getting OpenAI to use our Tool

There's some fiddly stuff to allow OpenAI "to call our tool"

What we actually do is give the LLM the opportunity to inform us that it wants 
us to run the tool.

Here's how the new chat function looks:

In [60]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    
    response = openai_client.chat.completions.create(
        model=gpt_model,
        messages=messages,
        tools=tools
        )

    if response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        response = handle_tool_call(message)
        
        messages.append(message)
        messages.append(response)
        
        response = openai_client.chat.completions.create(
            model= gpt_model, 
            messages=messages
            )
    
    return response.choices[0].message.content

In [61]:
# We have to write that function handle_tool_call

def handle_tool_call(message):
    
    tool_call = message.tool_calls[0]
    
    if tool_call.function.name == "get_ticket_price":
    
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city')
        price_details = get_ticket_price(city)
    
        response = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
    
    return response

In [62]:
# gr.ChatInterface(fn= chat).launch()

### Let's make a couple of improvements

Handling multiplw tool calls in 1 response  

Handling multiple tool calls 1 after another

In [63]:
def chat(message, history):
    
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    
    response = openai_client.chat.completions.create(
        model= gpt_model, 
        messages=messages, 
        tools=tools
    )

    if response.choices[0].finish_reason=="tool_calls":
        
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        
        messages.append(message)
        messages.extend(responses)
        
        response = openai_client.chat.completions.create(
            model= gpt_model, 
            messages=messages
        )
    
    return response.choices[0].message.content

In [64]:
def handle_tool_calls(message):
    
    responses = []
    
    for tool_call in message.tool_calls:
    
        if tool_call.function.name == "get_ticket_price":
    
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
    
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })

        elif tool_call.function.name == "set_ticket_price":
            
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get("city")
            price = arguments.get("price")

            set_ticket_price(city, price)

            responses.append(
                {
                    "role": "tool",
                    "content": f"Ticket price for {city} update to ${price}",
                    "tool_call_id" : tool_call.id
                }
            )
    
    return responses

In [65]:
# gr.ChatInterface(fn= chat).launch()

In [66]:
def chat(message, history):

    history = [{"role": h["role"],"content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]

    response = openai_client.chat.completions.create(
        model= gpt_model,
        messages= messages,
        tools= tools
    )

    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message

        responses = handle_tool_calls(message)
        
        messages.append(message)
        messages.extend(responses)

        response = openai_client.chat.completions.create(
            model= gpt_model,
            messages= messages,
            tools= tools
        )

    return response.choices[0].message.content

In [67]:
# gr.ChatInterface(fn= chat).launch()

In [81]:
def normalize_city(city):
    return city.strip().lower()

In [68]:
import sqlite3

In [69]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute("CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)")
    conn.commit()

In [82]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush= True)
    
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT price FROM prices WHERE city = ?", (normalize_city(city), ))
        result = cursor.fetchone()
        return f"Ticket price for city is ${result[0]}" if result else f"No price data available for {city}"

In [71]:
get_ticket_price("london")

DATABASE TOOL CALLED: Getting price for london


'Ticket price for city is $799.0'

In [83]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?", (normalize_city(city), price, price))
        conn.commit()

In [73]:

ticket_prices = {"london": 799, "paris": 899, "tokyo": 1420, "sydney": 2999}

In [84]:
for city, price in ticket_prices.items():
    set_ticket_price(city, price)


In [86]:
get_ticket_price("Tokyo")

DATABASE TOOL CALLED: Getting price for Tokyo


'Ticket price for city is $1420.0'

In [76]:
# gr.ChatInterface(fn= chat).launch()

### Exercise
Add a tool to set the price of a ticket!

In [87]:
set_price_function = {
    "name": "set_ticket_price",
    "description": "Set or update the ticket price for a given city in the SQL database.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "The destination city for which the ticket price should be set or updated."
            },
            "price": {
                "type": "number",
                "description": "The new ticket price for the city, stored as REAL in the database."
            }
        },
        "required": ["city", "price"],
        "additionalProperties": False
    }
}

In [88]:
tools = [
    {"type": "function", "function": price_function},
    {"type": "function", "function": set_price_function}
]

In [89]:
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'set_ticket_price',
   'description': 'Set or update the ticket price for a given city in the SQL database.',
   'parameters': {'type': 'object',
    'properties': {'city': {'type': 'string',
      'description': 'The destination city for which the ticket price should be set or updated.'},
     'price': {'type': 'number',
      'description': 'The new ticket price for the city, stored as REAL in the database.'}},
    'required': ['city', 'price'],
    'additionalProperties': False}}}]

In [92]:
# gr.ChatInterface(fn= chat).launch()